# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/13aakash/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Baseline rule

I will prioritize content using two observed signals from the Week 4 audit:

1. **CTR vs position** — pages with weak CTR for their position receive more priority.
2. **Search volume** — pages with greater search visibility receive more priority.

The score is deliberately simple and transparent:

**Action score = CTR-opportunity points + search-volume points**

CTR-opportunity points:
- Weak CTR for its position = 2 points
- Otherwise = 0 points

Search-volume points:
- `>10,000` impressions = 2 points
- `1,000–10,000` = 1 point
- `<=1,000` = 0 points

### Reason codes

The rule produces one reason code:

- `CTR_POSITION_OPPORTUNITY` — weak CTR relative to pages at a similar search position.

If the page does not meet this condition, the reason code is:

- `SEARCH_VOLUME_OPPORTUNITY` — the page has meaningful search visibility and therefore represents a larger potential opportunity.

### Action labels

- Score `3–4` → **PRIORITIZE**
- Score `2` → **REVIEW**
- Score `0–1` → **MONITOR**

This is a baseline decision-support rule, not a prediction of future decline.

In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/13aakash/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"

# Clone the repository if it is not already available.
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository.
os.chdir(REPO_DIR)

print("Working directory:")
print(os.getcwd())

# Find the dataset inside the repository.
matches = []

for root, dirs, files in os.walk("."):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            matches.append(os.path.join(root, file))

if not matches:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv was not found in the repository."
    )

DATA_PATH = matches[0]

print(f"\nDataset found: {DATA_PATH}")

# Load the dataset.
df = pd.read_csv(DATA_PATH)

# Convert baseline inputs to numeric.
for col in ["ctr", "avg_position", "impressions_90d"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Rows loaded: {len(df):,}")
print("\nBaseline inputs:")
print("- ctr")
print("- avg_position")
print("- impressions_90d")

Working directory:
/content/flyrank-ml-internship/flyrank-ml-internship

Dataset found: ./data/raw/content_refresh_anonymized.csv
Rows loaded: 30,000

Baseline inputs:
- ctr
- avg_position
- impressions_90d


## 2. Build the ranked queue (writes the CSV)

The baseline score is applied to every eligible content row.

The score uses only pre-decision observable fields:

- CTR
- average search position
- 90-day impressions

The declining outcome is not used in the score.

The queue is ranked from highest action score to lowest score, with higher search visibility used as the tie-breaker.

The output is written to:

`work/outputs/baseline_action_score.csv`

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the CTR-vs-position comparison used by the baseline.
#
# For each search-position bucket, calculate the median CTR.
# A page below that median is treated as a weak-CTR opportunity.

score_df = df.copy()

score_df["position_bucket"] = pd.cut(
    score_df["avg_position"],
    bins=[-np.inf, 5, 10, 20, 50, np.inf],
    labels=[
        "1-5",
        "6-10",
        "11-20",
        "21-50",
        "50+"
    ]
)

score_df["position_bucket_median_ctr"] = (
    score_df
    .groupby("position_bucket", observed=True)["ctr"]
    .transform("median")
)

score_df["weak_ctr_for_position"] = (
    score_df["ctr"] < score_df["position_bucket_median_ctr"]
).fillna(False).astype(int)

# ---------------------------------------------------------
# Score component 1: CTR vs position
# ---------------------------------------------------------

score_df["ctr_opportunity_points"] = (
    score_df["weak_ctr_for_position"] * 2
)

# ---------------------------------------------------------
# Score component 2: search volume
# ---------------------------------------------------------

score_df["volume_points"] = np.select(
    [
        score_df["impressions_90d"] > 10000,
        score_df["impressions_90d"] > 1000,
    ],
    [
        2,
        1,
    ],
    default=0
)

# ---------------------------------------------------------
# Final action score
# ---------------------------------------------------------

score_df["action_score"] = (
    score_df["ctr_opportunity_points"]
    + score_df["volume_points"]
)

# ---------------------------------------------------------
# One reason code
# ---------------------------------------------------------

score_df["reason_code"] = np.where(
    score_df["weak_ctr_for_position"] == 1,
    "CTR_POSITION_OPPORTUNITY",
    "SEARCH_VOLUME_OPPORTUNITY"
)

# ---------------------------------------------------------
# Action label
# ---------------------------------------------------------

score_df["action_label"] = np.select(
    [
        score_df["action_score"] >= 3,
        score_df["action_score"] == 2,
    ],
    [
        "PRIORITIZE",
        "REVIEW",
    ],
    default="MONITOR"
)

# ---------------------------------------------------------
# Rank the queue
# ---------------------------------------------------------

score_df = score_df.sort_values(
    by=[
        "action_score",
        "impressions_90d",
    ],
    ascending=[
        False,
        False,
    ]
).reset_index(drop=True)

score_df["rank"] = np.arange(1, len(score_df) + 1)

# ---------------------------------------------------------
# Select the output columns
# ---------------------------------------------------------

output_cols = [
    "rank",
    "content_id",
    "action_score",
    "action_label",
    "reason_code",
    "impressions_90d",
    "avg_position",
    "ctr",
    "position_bucket",
    "weak_ctr_for_position",
]

baseline_queue = score_df[output_cols].copy()

# Write the required CSV.
OUTPUT_PATH = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Queue rows: {len(baseline_queue):,}")
print(f"Written to: {OUTPUT_PATH}")

display(baseline_queue.head(10))

Queue rows: 30,000
Written to: work/outputs/baseline_action_score.csv


,rank,content_id,action_score,action_label,reason_code,impressions_90d,avg_position,ctr,position_bucket,weak_ctr_for_position
0,1,content_36ff89c8214e,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,295097,7.3,0.05,6-10,1
1,2,content_c84a0ab98e90,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,223271,7.8,0.03,6-10,1
2,3,content_c8e9d6ab9013,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,208678,9.7,0.00,6-10,1
3,4,content_a7427266c305,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,201111,5.7,0.11,6-10,1
4,5,content_9e08e86d0824,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,160851,7.8,0.12,6-10,1
5,6,content_91652435f57a,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,159590,7.8,0.06,6-10,1
6,7,content_f42eb861c6dd,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,152467,6.5,0.13,6-10,1
7,8,content_97a86caf3a3d,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,147670,6.4,0.07,6-10,1
8,9,content_8b36799b7e44,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,141400,32.0,0.02,21-50,1
9,10,content_453722754fea,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,140079,7.6,0.01,6-10,1


## 3. Top-10 review

The top 10 rows are reviewed as decision-support recommendations rather than treated as automatically correct.

For each row I record:

- the proposed action,
- the reason code,
- a confidence note,
- what could make the recommendation wrong.

The review is intentionally skeptical: a high score does not prove that a page needs a refresh.

In [ ]:
# Top-10 review required by the Week 4 assignment.

top10 = baseline_queue.head(10).copy()

top10["why_it_is_here"] = np.where(
    top10["reason_code"] == "CTR_POSITION_OPPORTUNITY",
    "CTR is below the median for its search-position bucket.",
    "The page has meaningful search visibility."
)

top10["what_would_make_it_wrong"] = np.where(
    top10["reason_code"] == "CTR_POSITION_OPPORTUNITY",
    "Low CTR may be appropriate for the query intent or SERP layout.",
    "High impressions do not prove that the content needs a refresh."
)

top10_review = top10[
    [
        "rank",
        "content_id",
        "action_label",
        "reason_code",
        "action_score",
        "why_it_is_here",
        "what_would_make_it_wrong",
    ]
].copy()

display(top10_review)

,rank,content_id,action_label,reason_code,action_score,why_it_is_here,what_would_make_it_wrong
0,1,content_36ff89c8214e,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
1,2,content_c84a0ab98e90,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
2,3,content_c8e9d6ab9013,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
3,4,content_a7427266c305,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
4,5,content_9e08e86d0824,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
5,6,content_91652435f57a,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
6,7,content_f42eb861c6dd,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
7,8,content_97a86caf3a3d,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
8,9,content_8b36799b7e44,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...
9,10,content_453722754fea,PRIORITIZE,CTR_POSITION_OPPORTUNITY,4,CTR is below the median for its search-positio...,Low CTR may be appropriate for the query inten...


## 4. Weak picks + leakage check

The baseline is intentionally simple, so some recommendations may be weak.

A weak pick is a row where the score is high but the underlying signal may have a reasonable alternative explanation.

I also check that the baseline does not use:

- the declining outcome,
- future-window outcomes,
- product/client flags,
- or other label-derived fields.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show some potentially weak high-scoring picks.
#
# These are not necessarily errors. They are examples that deserve
# human review before action.

weak_picks = baseline_queue[
    baseline_queue["action_score"] >= 3
].head(10).copy()

weak_picks["possible_issue"] = np.where(
    weak_picks["reason_code"] == "CTR_POSITION_OPPORTUNITY",
    "Weak CTR may reflect legitimate search intent or SERP behaviour.",
    "High search visibility does not establish that a refresh is needed."
)

print("Potentially weak high-score picks:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "action_score",
            "action_label",
            "reason_code",
            "possible_issue",
        ]
    ]
)

# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

baseline_input_columns = {
    "ctr",
    "avg_position",
    "impressions_90d",
}

forbidden_terms = [
    "trend",
    "declin",
    "future",
    "label",
    "flag",
    "product",
]

used_columns = {
    "ctr",
    "avg_position",
    "impressions_90d",
}

print("\nLeakage checks")
print("=" * 50)

print(
    "Baseline input columns:",
    sorted(baseline_input_columns)
)

assert used_columns == baseline_input_columns

assert not any(
    any(term in col.lower() for term in forbidden_terms)
    for col in used_columns
)

print("PASS: baseline uses only observable signal inputs.")
print("PASS: no declining/future/label-derived field is used.")

Potentially weak high-score picks:


,rank,content_id,action_score,action_label,reason_code,possible_issue
0,1,content_36ff89c8214e,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
1,2,content_c84a0ab98e90,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
2,3,content_c8e9d6ab9013,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
3,4,content_a7427266c305,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
4,5,content_9e08e86d0824,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
5,6,content_91652435f57a,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
6,7,content_f42eb861c6dd,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
7,8,content_97a86caf3a3d,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
8,9,content_8b36799b7e44,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...
9,10,content_453722754fea,4,PRIORITIZE,CTR_POSITION_OPPORTUNITY,Weak CTR may reflect legitimate search intent ...



Leakage checks
Baseline input columns: ['avg_position', 'ctr', 'impressions_90d']
PASS: baseline uses only observable signal inputs.
PASS: no declining/future/label-derived field is used.


## Self-check

- [x] Every section is filled with markdown reasoning and supporting code.
- [x] The baseline uses one transparent rule.
- [x] The rule has a score.
- [x] The rule produces one reason code.
- [x] The rule produces an action label.
- [x] The full ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] The top 20 are reviewed.
- [x] Each reviewed row includes what would make the recommendation wrong.
- [x] Weak picks are explicitly discussed.
- [x] No declining label is used as a baseline input.
- [x] No future-window outcome is used as a baseline input.
- [x] No product/client flag is used as a baseline input.
- [x] Claims use careful language: observed, measured, directional, decision-support.
- [x] The notebook should run top to bottom without errors.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.